# 02. Track with MLflow

Moringa Masterclass: Machine Learning End to End

This is part 2 of 3. We repeat the training from notebook 1, but this time every run is logged with MLflow: parameters, metrics, and the trained model itself. Then we compare runs and register the better model as a version.

This is the part of the workflow that answers "which model is actually in production, and why" six months from now.

In [ ]:
%pip install -q scikit-learn pandas numpy mlflow

In [ ]:
import pandas as pd

df = pd.read_csv("data/Telco-Customer-Churn.csv")
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce").fillna(0)

target = "Churn"
X = df.drop(columns=["customerID", target])
y = (df[target] == "Yes").astype(int)

numeric_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object"]).columns.tolist()

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

## 1. Point MLflow at a local tracking store

In a real production setup this would point at a shared MLflow tracking server. For the live session we use a local SQLite database, `mlflow.db`, so everyone can run this without any extra infrastructure. Recent MLflow versions no longer support a plain folder (`file:./mlruns`) as the tracking backend, so we use `sqlite:///mlflow.db` instead.

In [ ]:
import mlflow

mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("telco-churn")

## 2. Wrap training in an MLflow run

Same pipeline as notebook 1. The only change is that we log parameters, metrics, and the fitted pipeline itself inside an `mlflow.start_run()` block.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, roc_auc_score, precision_score, recall_score

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
    ]
)

def train_and_log(run_name, classifier, params):
    with mlflow.start_run(run_name=run_name):
        pipeline = Pipeline(steps=[
            ("preprocessor", preprocessor),
            ("classifier", classifier),
        ])
        pipeline.fit(X_train, y_train)

        y_pred = pipeline.predict(X_test)
        y_proba = pipeline.predict_proba(X_test)[:, 1]

        metrics = {
            "precision": precision_score(y_test, y_pred),
            "recall": recall_score(y_test, y_pred),
            "f1": f1_score(y_test, y_pred),
            "roc_auc": roc_auc_score(y_test, y_proba),
        }

        mlflow.log_params(params)
        mlflow.log_metrics(metrics)
        mlflow.sklearn.log_model(pipeline, name="model")

        print(run_name, metrics)
        return pipeline, metrics

In [ ]:
log_reg_pipeline, log_reg_metrics = train_and_log(
    "logistic_regression",
    LogisticRegression(max_iter=1000, random_state=42),
    {"model_type": "logistic_regression", "max_iter": 1000},
)

In [ ]:
rf_pipeline, rf_metrics = train_and_log(
    "random_forest",
    RandomForestClassifier(n_estimators=200, max_depth=8, random_state=42),
    {"model_type": "random_forest", "n_estimators": 200, "max_depth": 8},
)

## 3. Compare runs

We can pull runs back out of MLflow as a dataframe, which is often faster than opening the UI during a live demo. Then we launch the actual MLflow UI to look at it visually.

In [ ]:
runs = mlflow.search_runs(experiment_names=["telco-churn"])
runs[["tags.mlflow.runName", "metrics.f1", "metrics.roc_auc", "metrics.precision", "metrics.recall"]]

To open the MLflow UI: in a terminal, from this folder, run

```
mlflow ui --backend-store-uri sqlite:///mlflow.db
```

then open the URL it prints (usually http://127.0.0.1:5000). You will see both runs side by side, with their parameters, metrics, and the logged model artifact for each.

In Colab, the equivalent is to run the same command in a cell with `!`, then use a tunnel (e.g. `ngrok`) to view it, or simply rely on the `runs` dataframe above for the live session and demo the local UI from the presenter's machine.

## 4. Register the better model

Based on F1 and ROC-AUC, we register the stronger run as a named model version. This is the step that turns a one-off trained model into a versioned artifact other people and systems can reference by name.

In [ ]:
best_run_name = "random_forest" if rf_metrics["f1"] >= log_reg_metrics["f1"] else "logistic_regression"
print("Registering:", best_run_name)

runs_sorted = mlflow.search_runs(
    experiment_names=["telco-churn"],
    filter_string=f"tags.mlflow.runName = '{best_run_name}'",
    order_by=["start_time DESC"],
)
best_run_id = runs_sorted.iloc[0]["run_id"]

model_uri = f"runs:/{best_run_id}/model"
registered = mlflow.register_model(model_uri=model_uri, name="telco-churn-classifier")
print(registered)

## Recap

Both models are now logged with their parameters, metrics, and artifacts, and the better one is registered as `telco-churn-classifier`. If someone asks in six months which model is live and why it was chosen over the alternative, this is where the answer lives.

Next: `03_explain_shap_gemini.ipynb`, where we take individual predictions from the registered model, compute SHAP values, and use Gemini to turn those into plain-language explanations.